# Qwen Dialogue — Phase B (audio only)

**Phase A (local):** `python run_script_agents.py your_script.txt` → `output/lines.json`

**Voice refs (local, before upload):**
1. `python make_voices_refs.py` → warm **identity** `male_ref.wav` / `female_ref.wav` (living default, not flat)
2. `python make_emotion_refs.py` → optional tilts `male_curious.wav`, `female_warm.wav`, … (`config/emotions.yaml`)

**This notebook:** render episode from finished files only. **No OpenAI / Director.**

**Upload into Drive `inputs/`:**
1. `lines.json`
2. `male_ref.wav` + `female_ref.wav` (warm identity / code key `neutral`)
3. Emotion tilts: `male_*.wav` / `female_*.wav` (curious, warm, …)
4. `ref_texts.json` (identity + `emotions` block)

Survives disconnect: everything under `MyDrive/voice-generation/<RUN_NAME>/`.
Generate skips clips that already exist. Picks clone ref by `line.emotion` (missing → `warm`).


In [ ]:
# 1. GPU check
import torch

assert torch.cuda.is_available(), "Runtime > Change runtime type > T4 GPU"

gpu_name = torch.cuda.get_device_name(0)
capability = torch.cuda.get_device_capability(0)
BF16_OK = capability >= (8, 0)
DTYPE = torch.bfloat16 if BF16_OK else torch.float16

print(f"GPU: {gpu_name} | dtype={DTYPE}")
!nvidia-smi


In [ ]:
# 2. Mount Google Drive
from pathlib import Path
from google.colab import drive

drive.mount("/content/drive")

RUN_NAME = "chokhi_dhani_v1"  # change per episode
ROOT = Path("/content/drive/MyDrive/voice-generation") / RUN_NAME
INPUT_DIR = ROOT / "inputs"
CLIPS_DIR = ROOT / "clips"
OUT_DIR = ROOT / "outputs"
for d in (INPUT_DIR, CLIPS_DIR, OUT_DIR):
    d.mkdir(parents=True, exist_ok=True)

print(f"ROOT = {ROOT}")
print(f"inputs: {sorted(p.name for p in INPUT_DIR.iterdir())}")
print(f"clips:  {len(list(CLIPS_DIR.glob('*.wav')))} wav(s)")


In [ ]:
# 3. Install deps (no openai)
!pip install -q qwen-tts soundfile pydub
print("Installed.")


In [ ]:
# 4. Upload inputs (skipped if already on Drive)
import json
from google.colab import files

needed = ["lines.json", "male_ref.wav", "female_ref.wav", "ref_texts.json"]
have = {p.name for p in INPUT_DIR.iterdir()}
missing = [n for n in needed if n not in have]

if missing:
    print("Upload required (+ optional emotion wavs male_curious.wav, ...):", missing)
    uploaded = files.upload()
    for name, data in uploaded.items():
        (INPUT_DIR / name).write_bytes(data)
else:
    print("Core inputs already on Drive — skipping upload.")

for n in needed:
    assert (INPUT_DIR / n).exists(), f"Missing {INPUT_DIR / n}"

LINES = json.loads((INPUT_DIR / "lines.json").read_text(encoding="utf-8"))
REF_TEXTS = json.loads((INPUT_DIR / "ref_texts.json").read_text(encoding="utf-8"))
REF_TEXT_M, REF_TEXT_F = REF_TEXTS["male"], REF_TEXTS["female"]

emo_files = sorted(
    p.name for p in INPUT_DIR.glob("*_*.wav")
    if p.name not in ("male_ref.wav", "female_ref.wav")
    and (p.name.startswith("male_") or p.name.startswith("female_"))
)
print(f"lines.json: {len(LINES)} lines")
print(f"emotion wavs: {emo_files or '(none — fall back to identity)'}")
print(f"M identity: {REF_TEXT_M[:60]}...")
print(f"F identity: {REF_TEXT_F[:60]}...")
for L in LINES[:3]:
    print(f"  [{L['index']}] {L['speaker']} ({L.get('emotion')}) {L['text'][:70]}...")


In [ ]:
# 5. Load Qwen3-TTS Base + clone prompts (identity + emotion banks)
import time
import torch
from qwen_tts import Qwen3TTSModel

EMOTION_BANK = ("curious", "playful", "excited", "warm", "wonder", "explanatory")
SPEAKER_KEYS = {"M": "male", "F": "female"}
SAMPLE_FALLBACK = {
    "curious": "Wait… what is that over there? Is that part of the tradition too?",
    "playful": "Why not both? Come on — I dare you to try it with me.",
    "excited": "Okay — you have to see this! This is the moment right here!",
    "warm": "Yeah… this feels really special, doesn't it? I'm glad we're here.",
    "wonder": "This… is something else. Just look at it for a second.",
    "explanatory": "So here's the thing — they built it this way so visitors can feel the culture up close.",
}


def resolve_ref(speaker, emotion):
    sk = SPEAKER_KEYS[speaker]
    emo = (emotion or "neutral").lower()
    identity = INPUT_DIR / f"{sk}_ref.wav"
    identity_text = REF_TEXTS[sk]
    if emo != "neutral":
        cand = INPUT_DIR / f"{sk}_{emo}.wav"
        if cand.exists():
            bank = (REF_TEXTS.get("emotions") or {}).get(sk) or {}
            text = bank.get(emo) or SAMPLE_FALLBACK.get(emo, identity_text)
            return cand, text
    return identity, identity_text


print("Loading Qwen3-TTS Base...")
t0 = time.time()
qwen = Qwen3TTSModel.from_pretrained(
    "Qwen/Qwen3-TTS-12Hz-1.7B-Base",
    device_map="cuda",
    dtype=DTYPE,
    attn_implementation="sdpa",
)
print(f"Loaded in {time.time() - t0:.1f}s")

CLONE = {}
for speaker in ("M", "F"):
    for emotion in ("neutral", *EMOTION_BANK):
        path, text = resolve_ref(speaker, emotion)
        id_path, _ = resolve_ref(speaker, "neutral")
        if emotion != "neutral" and path.resolve() == id_path.resolve():
            CLONE[(speaker, emotion)] = CLONE[(speaker, "neutral")]
            print(f"  prompt {speaker}/{emotion} <- identity fallback")
            continue
        CLONE[(speaker, emotion)] = qwen.create_voice_clone_prompt(
            ref_audio=str(path),
            ref_text=text,
            x_vector_only_mode=False,
        )
        print(f"  prompt {speaker}/{emotion} <- {path.name}")

print(f"GPU mem: {torch.cuda.memory_allocated()/1e9:.2f} GB")
!nvidia-smi
print("Ready.")


In [ ]:
# 6. Generate each line (skips existing clips on Drive)
import json
import time
import numpy as np
import soundfile as sf
import torch

ONLY = None       # e.g. [1, 2, 3] for a preview
FORCE_REGEN = False
DEFAULT_EMOTION = "warm"  # missing/unknown → warm tilt; "neutral" = identity ref

lines = json.loads((INPUT_DIR / "lines.json").read_text(encoding="utf-8"))
if ONLY is not None:
    lines = [L for L in lines if L["index"] in ONLY]

TEMP_BY_EMOTION = {
    "neutral": 0.88,  # living identity — warm & present, not flat
    "curious": 0.9,
    "playful": 0.95,
    "excited": 0.95,
    "warm": 0.9,
    "wonder": 0.88,
    "explanatory": 0.82,
}

print(f"Generating into {CLIPS_DIR}")
for L in lines:
    out = CLIPS_DIR / f"{L['index']:03d}_{L['speaker']}.wav"
    if out.exists() and not FORCE_REGEN:
        print(f"skip {out.name}")
        continue
    emotion = (L.get("emotion") or DEFAULT_EMOTION).lower()
    prompt = CLONE.get((L["speaker"], emotion)) or CLONE[(L["speaker"], "neutral")]
    temp = TEMP_BY_EMOTION.get(emotion, 0.9)
    print(f"[{L['index']}] {L['speaker']} ({emotion}) temp={temp}")
    print(f"  {L['text'][:90]}...")
    t0 = time.time()
    torch.manual_seed(42 + int(L["index"]))
    torch.cuda.synchronize()
    wavs, sr = qwen.generate_voice_clone(
        text=L["text"],
        language="English",
        voice_clone_prompt=prompt,
        temperature=temp,
    )
    torch.cuda.synchronize()
    elapsed = time.time() - t0
    w = wavs[0]
    audio = w.detach().cpu().numpy() if hasattr(w, "detach") else np.asarray(w)
    audio = np.asarray(audio, dtype=np.float32).squeeze()
    sf.write(out, audio, sr)
    dur = len(audio) / float(sr)
    print(f"  -> {out.name}  audio={dur:.1f}s  wall={elapsed:.1f}s  ({elapsed/max(dur,0.01):.1f}x)")
    if elapsed > 180:
        print("  WARNING: >3 min/line is abnormal on T4 — check nvidia-smi GPU-Util.")
    if dur > 90:
        print("  WARNING: clip >90s — possible runaway generation.")

print("Generation done.")


In [ ]:
# 7. Stitch with lines.json pauses + silence trim + loudnorm
import json
import subprocess
import warnings
from pydub import AudioSegment
from pydub.silence import detect_nonsilent
import IPython.display as ipd

warnings.filterwarnings("ignore", category=SyntaxWarning, module=r"pydub.*")

LINES = json.loads((INPUT_DIR / "lines.json").read_text(encoding="utf-8"))


def trim_silence(audio, silence_thresh=-40, min_silence_len=250, padding_ms=50):
    if len(audio) == 0:
        return audio
    nonsilent = detect_nonsilent(
        audio, min_silence_len=min_silence_len, silence_thresh=silence_thresh
    )
    if not nonsilent:
        return audio
    start = max(nonsilent[0][0] - padding_ms, 0)
    end = min(nonsilent[-1][1] + padding_ms, len(audio))
    return audio[start:end]


episode = AudioSegment.empty()
missing = []
for L in LINES:
    path = CLIPS_DIR / f"{L['index']:03d}_{L['speaker']}.wav"
    if not path.exists():
        missing.append(path.name)
        continue
    clip = trim_silence(AudioSegment.from_wav(path))
    pb = int(L.get("pause_before_ms") or 0)
    pa = int(L.get("pause_after_ms") or 400)
    if pb > 0:
        episode += AudioSegment.silent(duration=pb)
    episode += clip
    episode += AudioSegment.silent(duration=pa)

assert len(episode) > 0, f"No clips. Missing: {missing}"
if missing:
    print(f"WARNING: skipped missing: {missing}")

raw_wav = OUT_DIR / "episode_raw.wav"
EPISODE_PATH = OUT_DIR / "episode.wav"
episode.export(raw_wav, format="wav")
subprocess.run(
    [
        "ffmpeg", "-y", "-i", str(raw_wav),
        "-af", "loudnorm=I=-16:TP=-1.5:LRA=11",
        str(EPISODE_PATH),
    ],
    check=True,
    capture_output=True,
)
print(f"Wrote {EPISODE_PATH} ({len(episode)/60000:.1f} min)")
ipd.Audio(str(EPISODE_PATH))


In [ ]:
# 8. Download (optional — already on Drive)
from google.colab import files

files.download(str(EPISODE_PATH))
print(f"Also on Drive: {OUT_DIR}")


## After disconnect

1. Cells 1–3 (GPU, Drive, pip)
2. Cell 4 skips upload if files exist
3. Cell 5 reloads Qwen (required)
4. Cell 6 skips finished clips
5. Cell 7 stitch

**Phase A again** (new script): run locally `python run_script_agents.py …`, upload new `lines.json`, delete old `clips/*.wav` if texts changed, regenerate.

**New voice refs:** re-upload identity + emotion wavs + `ref_texts.json`, then FORCE_REGEN or delete clips.
